# AML LoT - Patient intensity and line of therapy

Split from `AML_LOT_VAL.ipynb`. Fully commented-out cells removed.

# SPARK SESSION

In [ ]:
import findspark
import pandas as pd
findspark.init ()
import pyspark
from pyspark.sql import SparkSession

spark= SparkSession.builder\
              .config('spark.executor.instances','20')\
              .config('spark.driver.memory','5g')\
              .config('spark.executor.memory','20g')\
              .config('spark.executor.cores','5')\
              .config('spark.executor.memoryOverhead','4g')\
              .appName("AML_KA")\
              .master("yarn")\
              .getOrCreate()

In [ ]:
spark.sparkContext.addPyFile("pldlib.egg")
from pldlib import stencil

# GLOBAL VARIABLES

In [ ]:
cws = 'Z_ABV_CWS_MABI_ONC_ANALYTICS' 

# HCP-CHILD
reltio_tbl = 'abv_mhcd_synd.mhcd_rltn_all_org_customr_wkly_tbl'
heme_account='{cws}.heme_account'.format(cws=cws) # UPLOADED BY COE

tx_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_UPDATED_TX_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_tbl_final_name = '{cws}.MABI_AML_REGIMEN_REVAMP_UPDATED_TX_TBL_FINAL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

sob_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_SOB_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
episode_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_EPISODE_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
regimen_tbl_name = '{cws}.MABI_AML_REGIMEN_REVAMP_BASIC_REGIMEN_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

patient_group = '{cws}.MABI_AML_REGIMEN_REVAMP_PATIENT_COHORT_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

# LoT IC INELIGIBLE
ic_inelig_final = '{cws}.MABI_AML_IC_INELIG_LOT_FINAL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

# LOT IC ELIGIBLE
ic_elig_final = '{cws}.MABI_AML_IC_ELIG_LOT_FINAL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws) # LOT table IC ELIG
pat_comb_lot = '{cws}.MABI_AML_PATS_COMB_LOT_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)  # UNION OF BOTH LOT FINAL TABLES
pat_comb_lot_rr = '{cws}.MABI_AML_PATS_LOT_RR_12M_UPD_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)    

pat_fst_dx_tx = '{cws}.MABI_AML_LOT_PAT_FST_DX_TX_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

dx_lookback_elig = '{cws}.MABI_AML_LOT_PAT_DX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_lookback_elig = '{cws}.MABI_AML_LOT_PAT_TX_MX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_lookback_elig_rx = '{cws}.MABI_AML_LOT_PAT_TX_RX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_txl_mx_elig = '{cws}.MABI_AML_LOT_PAT_TX_MX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_txl_rx_elig = '{cws}.MABI_AML_LOT_PAT_TX_RX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

tx_lookback_elig_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_MX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_lookback_elig_rx_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_RX_LB_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_txl_mx_elig_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_MX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
tx_txl_rx_elig_ven = '{cws}.MABI_AML_LOT_PAT_VEN_TX_RX_TXL_ELIG_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

lot_regimen_group = '{cws}.MABI_AML_PATS_LOT_GROUPING_TBL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)
patient_intensity_based_on_age_n_product = '{cws}.PATIENT_INTENSITY_BASED_ON_AGE_N_PRODUCT_v2'.format(cws=cws)

#HCP AFFILIATIONS
hcp_affiliations = '{cws}.MABI_AML_HCP_AFFILIATIONS'.format(cws=cws)

# LoT TABLE ELIGIBILITY CHECK
elig_flags_final='{cws}.MABI_AML_LOT_PAT_ELIG_FLAGS_FINAL_BUSINESS_RULE_CHANGE_VAL_v2'.format(cws=cws)

#RR AML DX ONLY
aml_dx = ('C92.01', 'C92.02', 'C92.51', 'C92.52', 'C92.61', 'C92.62', 'C92.A1', 'C92.A2', 'C93.01', 'C93.02', 'C94.01', 'C94.21', 'C94.22')


#### IC ELIGIBLE / IC INELIGIBLE DEFINITION (config-driven)

`run.intensity_definition` in `config/common.yaml` chooses how patients are split into IC_ELIG and
IC_INELIG before lines of therapy are built:

- **legacy** (default): product-only. A patient is IC_ELIG if any regimen in the journey contains an
  IC-Eligible-only product, otherwise IC_INELIG. This is the production rule and the default
  everywhere. A legacy run writes exactly the same tables as before.
- **new**: age + product. Each patient is classified from the legacy line-1 product
  group. Any IC-Eligible product makes the patient IC_ELIG. A line 1 of only HMA/IDH products makes
  them IC_INELIG. Otherwise age at first diagnosis decides (<= 65 IC_ELIG, > 65 IC_INELIG); with no
  age the patient is IC_INELIG.

A **new** run works in two passes:

1. Build lines with the legacy rule into scratch tables (`_P1LEGACY`).
2. Classify each patient with the new definition from that line 1, then rebuild the IC_ELIG /
   IC_INELIG lines, the combined LoT and the LoT grouping using the new classes.

Outputs carry `run.new_definition_tag` (`_NEWDEF`), so the legacy tables are never overwritten.

In [ ]:
import sys
sys.path.append('../../config')
from load_config import load_config
import intensity_definition as idef

cfg = load_config('intensity_lot.yaml')
INTENSITY_DEFINITION = idef.definition(cfg)      # 'legacy' (default) or 'new'; anything else raises
DEF_TAG = idef.tag(cfg)                          # '' for legacy, '_NEWDEF' for new
print('IC eligible / ineligible definition:', idef.label(cfg))

# Final output names for this run. Legacy keeps the names set in GLOBAL VARIABLES above.
final_patient_group = patient_group + DEF_TAG
final_ic_inelig_final = ic_inelig_final + DEF_TAG
final_ic_elig_final = ic_elig_final + DEF_TAG
final_pat_comb_lot = pat_comb_lot + DEF_TAG
final_lot_regimen_group = lot_regimen_group + DEF_TAG
final_patient_intensity_tbl = patient_intensity_based_on_age_n_product + DEF_TAG

if INTENSITY_DEFINITION == 'new':
    # Pass 1 runs the unchanged legacy cells into scratch tables.
    patient_group = idef.scratch(patient_group)
    ic_inelig_final = idef.scratch(ic_inelig_final)
    ic_elig_final = idef.scratch(ic_elig_final)
    pat_comb_lot = idef.scratch(pat_comb_lot)
    lot_regimen_group = idef.scratch(lot_regimen_group)
print('Pass-1 cohort / combined LoT tables:', patient_group, pat_comb_lot)

# PATIENT COHORT

In [ ]:
patient_group

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {patient_group}'''.format(patient_group=patient_group))

Changes Made

In [ ]:
# INQOVI and ONUREG are classified as IC Ineligible products.
# Rydapt and Mylotarg are classified as IC Eligible only.

spark.sql('''
CREATE TABLE {patient_group} AS
(
    SELECT A.*,
           CASE WHEN PATIENT_INTENSITY = 'IC_INELIG' THEN 'IC_INELIG'
           ELSE 'IC_ELIG'
           END AS PATIENT_INTENSITY_GROUP
    FROM
    (
        SELECT *
        FROM
        (
            SELECT *,
            CASE
                WHEN LOW_INTENSITY = 1 THEN 'IC_INELIG'
                ELSE 'IC_ELIG'
            END AS PATIENT_INTENSITY
            FROM
            (
                SELECT PATIENT_GID_RGMN, MIN(LOW_INTENSITY_INT) AS LOW_INTENSITY
                FROM
                (

                    --Simplified the code as well
          
                    SELECT *, LOW_INTENSITY AS LOW_INTENSITY_INT
                    
                    
                    --SELECT *,
                    --CASE    
                    --    WHEN LOW_INTENSITY = 0 THEN 0
                    --    WHEN LOW_INTENSITY = 0 AND RYD_MYL_FLG = 1 THEN 1
                    --    ELSE LOW_INTENSITY
                    --END AS LOW_INTENSITY_INT


                    FROM
                    (
                        SELECT *,
                        CASE 
          
                            WHEN REGIMEN LIKE '%ARSENIC_TRIOXIDE%' THEN 0
                            WHEN REGIMEN LIKE '%CLADRIBINE%' THEN 0
                            WHEN REGIMEN LIKE '%CLOFARABINE%' THEN 0
                            WHEN REGIMEN LIKE '%CYCLOPHOSPHAMIDE%' THEN 0
                            WHEN REGIMEN LIKE '%S-DAC%' THEN 0
                            WHEN REGIMEN LIKE '%HI-DAC%' THEN 0
                            WHEN REGIMEN LIKE '%DAUNORUBICIN%' THEN 0
                            WHEN REGIMEN LIKE '%ETOPOSIDE%' THEN 0
                            WHEN REGIMEN LIKE '%FILGRASTIM%' THEN 0
                            WHEN REGIMEN LIKE '%FLUDARABINE%' THEN 0
                            WHEN REGIMEN LIKE '%IDARUBICIN%' THEN 0
                            --WHEN REGIMEN LIKE '%INQOVI%' THEN 0  --NOW INQOVI CONSIDERED AS IC INELIG
                            WHEN REGIMEN LIKE '%MITOXANTRONE%' THEN 0
                            WHEN REGIMEN LIKE '%MYLOTARG%' THEN 0
                            WHEN REGIMEN LIKE '%NEXAVAR%' THEN 0
                            --WHEN REGIMEN LIKE '%ONUREG%' THEN 0  --NOW ONUREG CONSIDERED AS IC INELIG
                            WHEN REGIMEN LIKE '%RYDAPT%' THEN 0
                            WHEN REGIMEN LIKE '%VANFLYTA%' THEN 0
                            WHEN REGIMEN LIKE '%VINCRISTINE%' THEN 0
                            WHEN REGIMEN LIKE '%VYXEOS%' THEN 0
                            WHEN REGIMEN LIKE '%RYDAPT%' THEN 0
                            WHEN REGIMEN LIKE '%MYLOTARG%' THEN 0
                            ELSE 1
                        END AS LOW_INTENSITY

                        --Special condition for Rydapt and Mylotarg removed since they are considered only as IC Elig
                        --,CASE 
                        --    WHEN REGIMEN = ('RYDAPT, VENCLEXTA') OR REGIMEN = ('MYLOTARG, VENCLEXTA') 
                        --    OR REGIMEN = ('AZACITIDINE, RYDAPT') OR REGIMEN = ('DECITABINE, RYDAPT') 
                        --    OR REGIMEN = ('DECITABINE, MYLOTARG') OR REGIMEN = ('AZACITIDINE, MYLOTARG') THEN 1 
                        --    ELSE 0
                        --END AS RYD_MYL_FLG

                        FROM 
                        (
                            SELECT DISTINCT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE
                            FROM {regimen_tbl_name}
                        )A
                    )A
                )A
                GROUP BY 1
            )A
        )A
    )A
)
'''.format(patient_group=patient_group,regimen_tbl_name=regimen_tbl_name,pat_fst_dx_tx=pat_fst_dx_tx))

In [ ]:
spark.sql('''
SELECT PATIENT_INTENSITY_GROUP, COUNT(DISTINCT PATIENT_GID_RGMN) AS NUM_PAT
FROM {patient_group}
GROUP BY 1
'''.format(patient_group=patient_group)).toPandas().head(20)

In [ ]:
spark.sql('''
SELECT DISTINCT FINAL_PRODUCT_NAME
FROM {tx_tbl_final_name}
WHERE PATIENT_GID IN
(
  SELECT DISTINCT PATIENT_GID_RGMN
  FROM {patient_group}
  WHERE PATIENT_INTENSITY_GROUP = 'IC_INELIG'
)
'''.format(tx_tbl_final_name=tx_tbl_final_name,patient_group=patient_group)).toPandas().head(20)

# REGIMEN CONSOLIDATION - IC INELIG

#### FINAL CONS - LOW INTENSITY + VALID LENGTH

Changes Made

In [ ]:
# Removing exception condition for Rydapt and smaller regimen dropping condition

ic_inelig_cons_final = spark.sql('''
SELECT *
FROM
(
    -- Smaller regimens were dropped earlier, now all the regimens are kept as it is
                                 
    SELECT *  
    --,CASE
      --  WHEN REGIMEN = 'RYDAPT' AND REGIMEN_LENGTH >= 14 THEN 1 --RYDAPT CLAIMS HAVE DOS OF 14/21 AS WELL
      --  WHEN REGIMEN_LENGTH >= 28 THEN 1
     --   ELSE 0
    --END AS SEL_FLAG
                                 
    FROM
    (   SELECT PATIENT_GID_RGMN AS PATIENT_GID,  REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE,
        --SELECT PATIENT_GID_RGMN AS PATIENT_GID, NEW_RNK, REGIMEN, LVL, REGIMEN_START_DATE, REGIMEN_END_DATE,
        DATEDIFF(REGIMEN_END_DATE, REGIMEN_START_DATE) AS REGIMEN_LENGTH
        --FROM ic_inelig_cons5
         FROM {regimen_tbl_name}
 									WHERE PATIENT_GID_RGMN IN
 									(
 										SELECT DISTINCT PATIENT_GID_RGMN
 										FROM {patient_group}
 										WHERE PATIENT_INTENSITY_GROUP = 'IC_INELIG'
 									)
    )A
)A
                                 
--WHERE SEL_FLAG = 1
                                 
'''.format(regimen_tbl_name=regimen_tbl_name, patient_group=patient_group))

ic_inelig_cons_final.createOrReplaceTempView('ic_inelig_cons_final')

#### LoT ASSIGNMENT

In [ ]:
ic_inelig_final

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {ic_inelig_final}'''.format(ic_inelig_final=ic_inelig_final))

Changes Made

In [ ]:
# Addition of Inqovi and Onureg as HMAs and removal of Rydapt and Mylotarg in Backbone Product hierarchy

spark.sql('''
CREATE TABLE {ic_inelig_final} AS
(	
	SELECT *,
	SUM(LINE_CHANGE) OVER (PARTITION BY PATIENT_GID ORDER BY REGIMEN_START_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS LOT
	FROM
	(
		SELECT *,
		CASE
			WHEN REG_NUM = 1 THEN 1
			WHEN PREV_GAP > 90 THEN 1
			WHEN PREV_GAP <= 90 THEN
				CASE
					WHEN BACKBONE = PREV_BACKBONE THEN 0
					WHEN BACKBONE LIKE '%VENCLEXTA%' AND PREV_BACKBONE LIKE '%HMA%' THEN 0
					WHEN BACKBONE LIKE '%HMA%' AND PREV_BACKBONE LIKE '%VENCLEXTA%' THEN 0
					ELSE 1 --WHEN BACKBONE <> PREV_BACKBONE THEN 1
				END
		END AS LINE_CHANGE
		FROM
		(
			SELECT *,
			LAG(BACKBONE) OVER (PARTITION BY PATIENT_GID ORDER BY REGIMEN_START_DATE) AS PREV_BACKBONE,
			DATEDIFF(REGIMEN_START_DATE, LAG(REGIMEN_END_DATE) OVER (PARTITION BY PATIENT_GID ORDER BY REGIMEN_START_DATE)) AS PREV_GAP
			FROM
			(
				SELECT PATIENT_GID, REGIMEN, 
				TO_DATE(REGIMEN_START_DATE) AS REGIMEN_START_DATE, 
				TO_DATE(REGIMEN_END_DATE) AS REGIMEN_END_DATE, REGIMEN_LENGTH,
				ROW_NUMBER() OVER (PARTITION BY PATIENT_GID ORDER BY REGIMEN_START_DATE) AS REG_NUM,
				CASE
          
					WHEN REGIMEN RLIKE 'VENCLEXTA' THEN 'VENCLEXTA'
					WHEN REGIMEN RLIKE 'TIBSOVO' THEN 'TIBSOVO'
					WHEN REGIMEN RLIKE 'REZLIDHIA' THEN 'REZLIDHIA'
					WHEN REGIMEN RLIKE 'IDHIFA' THEN 'IDHIFA'
					WHEN REGIMEN RLIKE 'XOSPATA' THEN 'XOSPATA'
					WHEN REGIMEN RLIKE 'DAURISMO' THEN 'DAURISMO'
					--WHEN REGIMEN RLIKE 'RYDAPT' THEN 'RYDAPT'   --REMOVING RYDAPT AND MYLOTARG FROM IC INELIG BACKBONE HIERARCHY
					--WHEN REGIMEN RLIKE 'MYLOTARG' THEN 'MYLOTARG'
					WHEN REGIMEN RLIKE 'L-DAC' THEN 'L-DAC'
					WHEN REGIMEN RLIKE 'AZACITIDINE' THEN 'HMA'
					WHEN REGIMEN RLIKE 'DECITABINE' THEN 'HMA'
					WHEN REGIMEN RLIKE 'INQOVI' THEN 'HMA'		--ADDING INQOVI AND ONUREG IN IC INELIG BACKBONE HIERARCHY
					WHEN REGIMEN RLIKE 'ONUREG' THEN 'HMA'
				END AS BACKBONE
				FROM ic_inelig_cons_final
			)A
		)A
	)
)
'''.format(ic_inelig_final=ic_inelig_final))

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID)
FROM {ic_inelig_final}
'''.format(ic_inelig_final=ic_inelig_final)).show()

#### LOT ASSIGNMENT - IC ELIG

In [ ]:
ic_elig_final

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {ic_elig_final}'''.format(ic_elig_final=ic_elig_final))

Changes Made

In [ ]:
# Inqovi and ONUREG are treated as HMAs

spark.sql('''
CREATE TABLE {ic_elig_final} AS
( 
	SELECT *,
	SUM(LINE_CHANGE) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS LOT
	FROM
	(
		SELECT *,
		CASE
			WHEN REG_NUM = 1 THEN 1
			WHEN PREV_GAP > 90 THEN 1
			WHEN PREV_GAP <= 90 THEN
				CASE
					WHEN BACKBONE = PREV_BACKBONE THEN 0	
					WHEN BACKBONE LIKE '%VENCLEXTA%' AND PREV_BACKBONE LIKE '%HMA%' THEN 0
					WHEN BACKBONE LIKE '%HMA%' AND PREV_BACKBONE LIKE '%VENCLEXTA%' THEN 0
					ELSE 1 											
				END
		END AS LINE_CHANGE
		FROM
		(
			SELECT *,
			LAG(BACKBONE) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS PREV_BACKBONE,
			DATEDIFF((REGIMEN_START_DATE), LAG(REGIMEN_END_DATE) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE)) AS PREV_GAP
			FROM
			(
				SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, 
				DATEDIFF((REGIMEN_END_DATE) , (REGIMEN_START_DATE)) AS REG_LEN,
				ROW_NUMBER() OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS REG_NUM,
				CASE
					WHEN REGIMEN LIKE '%HI-DAC%' OR REGIMEN LIKE '%S-DAC%' THEN 'S-DAC/HI-DAC'
					WHEN REGIMEN LIKE '%VANFLYTA%' THEN 'VANFLYTA'
					WHEN REGIMEN LIKE '%VYXEOS%' THEN 'VYXEOS'
					WHEN REGIMEN LIKE '%CYCLOPHOSPHAMIDE%' THEN 'CYCLOPHOSPHAMIDE'
					WHEN REGIMEN LIKE '%VINCRISTINE%' THEN 'VINCRISTINE'
					WHEN REGIMEN LIKE '%FLUDARABINE%' THEN 'FLUDARABINE'
					WHEN REGIMEN LIKE '%IDARUBICIN%' THEN 'IDARUBICIN'
					WHEN REGIMEN LIKE '%DAUNORUBICIN%' THEN 'DAUNORUBICIN'
					WHEN REGIMEN LIKE '%MITOXANTRONE%' THEN 'MITOXANTRONE'
					WHEN REGIMEN LIKE '%RYDAPT%' THEN 'RYDAPT'
					WHEN REGIMEN LIKE '%VENCLEXTA%' THEN 'VENCLEXTA'
					WHEN REGIMEN LIKE '%XOSPATA%' THEN 'XOSPATA'
					WHEN REGIMEN LIKE '%TIBSOVO%' THEN 'TIBSOVO'
					WHEN REGIMEN LIKE '%REZLIDHIA%' THEN 'REZLIDHIA'
					WHEN REGIMEN LIKE '%IDHIFA%' THEN 'IDHIFA'
					WHEN REGIMEN LIKE '%MYLOTARG%' THEN 'MYLOTARG'
					WHEN REGIMEN LIKE '%DAURISMO%' THEN 'DAURISMO'
					WHEN REGIMEN LIKE '%ONUREG%' THEN 'HMA'    --ONUREG AND INQOVI ARE TREATED AS HMAs
					WHEN REGIMEN LIKE '%INQOVI%' THEN 'HMA'
					WHEN REGIMEN LIKE '%NEXAVAR%' THEN 'NEXAVAR'
					WHEN REGIMEN LIKE '%ETOPOSIDE%' THEN 'ETOPOSIDE'
					WHEN REGIMEN LIKE '%CLADRIBINE%' THEN 'CLADRIBINE'
					WHEN REGIMEN LIKE '%CLOFARABINE%' THEN 'CLOFARABINE'
					WHEN REGIMEN LIKE '%AZACITIDINE%' OR REGIMEN LIKE '%DECITABINE%' THEN 'HMA'
					WHEN REGIMEN LIKE '%L-DAC%' THEN  'L-DAC'
					WHEN REGIMEN LIKE '%ARSENIC_TRIOXIDE%' THEN 'ARSENIC_TRIOXIDE' 
					ELSE NULL
				END AS BACKBONE
				FROM {regimen_tbl_name}
									WHERE PATIENT_GID_RGMN IN
									(
										SELECT DISTINCT PATIENT_GID_RGMN
										FROM {patient_group}
										WHERE PATIENT_INTENSITY_GROUP = 'IC_ELIG'
									)
			)A
		)A
	)	
)
'''.format(ic_elig_final=ic_elig_final, regimen_tbl_name=regimen_tbl_name, patient_group=patient_group))

In [ ]:
spark.sql('''
SELECT COUNT(DISTINCT PATIENT_GID_RGMN) 
FROM {ic_elig_final}
'''.format(ic_elig_final=ic_elig_final)).show()

# LOT TABLE FINAL - COMBINING IC ELIG AND INELIG PATIENTS

In [ ]:
spark.sql('''DROP TABLE IF EXISTS {pat_comb_lot}'''.format(pat_comb_lot=pat_comb_lot))

In [ ]:
spark.sql('''
CREATE TABLE {pat_comb_lot} AS
(
	SELECT * 
    FROM
	(
		SELECT PATIENT_GID_RGMN AS PATIENT_GID, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, BACKBONE, PREV_BACKBONE, REG_NUM, LINE_CHANGE, LOT, 'IC_ELIG' AS TYPE
		FROM {ic_elig_final} -- SOME PATIENTS MAY HAVE ARSENIC REGIMENS

		UNION
		
		SELECT PATIENT_GID, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, BACKBONE, PREV_BACKBONE, REG_NUM, LINE_CHANGE, LOT, 'IC_INELIG' AS TYPE
		FROM {ic_inelig_final}
	)   
)
'''.format(ic_elig_final=ic_elig_final, ic_inelig_final=ic_inelig_final, pat_comb_lot=pat_comb_lot))

#### Patient pool (re-created: used by the age + product intensity step)

In [ ]:
# New Patient Pool Created

patient_pool = spark.sql('''
SELECT distinct PATIENT_SK from
(
    select patient_sk, count(distinct mx_claim_id) as Count_of_claims
    FROM abv_val_ptd_onc_synd.dx_fact_curated_vw
    WHERE SOURCE_DIAGNOSIS_CODE_1 IN 
    (
        '205', '205.0', '205.00', '205.01', '205.02', 'C92', 'C92.9', 'C92.0', 'C92.00', 'C92.01', 
        'C92.02', 'C92.5', 'C92.50', 'C92.51', 'C92.52', 'C92.6', 'C92.60', 'C92.61', 'C92.62', 'C92.90', 
        'C92.A', 'C92.A0', 'C92.A1', 'C92.A2', 'C92.Z0', 'C93', 'C93.0', 'C93.00', 'C93.01', 'C93.02', 
        'C94.0', 'C94.00', 'C94.01', 'C94.2', 'C94.20', 'C94.21', 'C94.22'
    )
    and source_flag='SHA_PTD'
    group by 1
)
where count_of_claims>1
          ''')

patient_pool.createOrReplaceTempView('patient_pool')

#### New Patient Intensity Rule (Based on Age + Product)

New Code Blocks added post this for the new column to be added

Comnbining all the regimens in the LoT 

In [ ]:
LOT_TBL = spark.sql(''' 
select * 
          from {pat_comb_lot}

'''.format(pat_comb_lot=pat_comb_lot))

LOT_TBL.createOrReplaceTempView('LOT_TBL')

In [ ]:
# Converting to Pandas dataframe

lot = LOT_TBL.toPandas()

df_lot = pd.DataFrame(lot)

# Function to merge regimens while keeping comma-separated format
def merge_regimens(regimens):
    # Split by comma, strip spaces, flatten, deduplicate
    merged = set()
    for regimen in regimens:
        for drug in regimen.split(","):
            merged.add(drug.strip())
    
    # Sort alphabetically and join
    return ", ".join(sorted(merged))

# Group by PATIENT_GID and LOT
LOT_REGIMEN_GROUP = (
    df_lot.groupby(["PATIENT_GID", "LOT"])
    .agg(
        LOT_START_DATE=("REGIMEN_START_DATE", "min"),
        LOT_END_DATE=("REGIMEN_END_DATE", "max"),
        LOT_REGIMEN_GROUP=("REGIMEN", merge_regimens),
    )
    .reset_index()
)

print(LOT_REGIMEN_GROUP)


In [ ]:
spark_df_LOT_REGIMEN_GROUP = spark.createDataFrame(LOT_REGIMEN_GROUP)
spark_df_LOT_REGIMEN_GROUP.createOrReplaceTempView("spark_df_LOT_REGIMEN_GROUP")

In [ ]:
spark.sql(''' DROP TABLE IF EXISTS {lot_regimen_group} '''.format(lot_regimen_group=lot_regimen_group))


spark.sql('''CREATE TABLE  {lot_regimen_group} AS
           (
          SELECT * FROM spark_df_LOT_REGIMEN_GROUP
          )
          '''.format(lot_regimen_group=lot_regimen_group))



First Dx and Tx Creation

In [ ]:
# Creating First Dx Tx table

fst_dx_tx_tbl = spark.sql(f'''
SELECT *
FROM
(
    SELECT A.PATIENT_GID,
           B.FIRST_DX,
           A.FIRST_TX,
           A.LAST_TX
    FROM
    (
        SELECT PATIENT_GID,
               MIN(TO_DATE(regimen_start_date)) AS FIRST_TX,
               MAX(TO_DATE(regimen_end_date)) AS LAST_TX
        FROM {pat_comb_lot}
        GROUP BY PATIENT_GID
    ) A
    LEFT JOIN
    (
        SELECT PATIENT_SK AS PATIENT_GID,
               MIN(TO_DATE(CLAIM_DATE)) AS FIRST_DX
        FROM abv_val_ptd_onc_synd.dx_fact_curated_vw
        WHERE SOURCE_DIAGNOSIS_CODE_1 IN 
        (
            '205', '205.0', '205.00', '205.01', '205.02', 'C92', 'C92.9', 'C92.0', 'C92.00', 'C92.01', 
            'C92.02', 'C92.5', 'C92.50', 'C92.51', 'C92.52', 'C92.6', 'C92.60', 'C92.61', 'C92.62', 'C92.90', 
            'C92.A', 'C92.A0', 'C92.A1', 'C92.A2', 'C92.Z0', 'C93', 'C93.0', 'C93.00', 'C93.01', 'C93.02', 
            'C94.0', 'C94.00', 'C94.01', 'C94.2', 'C94.20', 'C94.21', 'C94.22'
        )
        
        GROUP BY PATIENT_SK
    ) B
    ON A.PATIENT_GID = B.PATIENT_GID
                      
) A
''')

fst_dx_tx_tbl.createOrReplaceTempView('fst_dx_tx_tbl')


New Patient Intensity Assignment

In [ ]:
patient_intensity_updated_interim_tbl = spark.sql(''' 

SELECT 
    a.PATIENT_GID, 
    a.LOT_REGIMEN_GROUP, 
    a.LOT_START_DATE, 
    A.LOT_END_DATE,
    b.SOURCE_PATIENT_HIPAA_BIRTH_YEAR, 
    c.FIRST_DX
FROM (
    SELECT 
        PATIENT_GID, LOT_START_DATE, LOT_END_DATE, 
        LOT_REGIMEN_GROUP
    FROM {lot_regimen_group}
    WHERE LOT = 1
) a
LEFT JOIN (
    SELECT DISTINCT 
        PATIENT_SK, 
        SOURCE_PATIENT_HIPAA_BIRTH_YEAR
    FROM abv_val_ptd_onc_venc_synd.all_claims_combined_fact_vw
    WHERE source_flag = 'SHA_PTD'
      AND source_market = 'ONC'
      --AND market_code = 'VENC_AML'
      and patient_sk in (
    select patient_sk
    from patient_pool
)


) b 
    ON a.PATIENT_GID = b.PATIENT_SK
LEFT JOIN fst_dx_tx_tbl c 
    ON a.PATIENT_GID = c.PATIENT_GID
                            

'''.format(lot_regimen_group=lot_regimen_group))

patient_intensity_updated_interim_tbl.createOrReplaceTempView('patient_intensity_updated_interim_tbl')

In [ ]:
updated_patient_intensity = patient_intensity_updated_interim_tbl.toPandas()

In [ ]:


# Define product sets
IC_ELIG_PRODUCTS = {
    "MYLOTARG","RYDAPT","ARSENIC_TRIOXIDE","CLADRIBINE","CLOFARABINE","CYCLOPHOSPHAMIDE",
    "DAUNORUBICIN","ETOPOSIDE","FILGRASTIM","FLUDARABINE","HI-DAC","IDARUBICIN",
    "MITOXANTRONE","NEXAVAR","S-DAC","VINCRISTINE","VANFLYTA","VYXEOS"
}

FORCED_INELIG_PRODUCTS = {"AZACITIDINE","DECITABINE","ONUREG","INQOVI","TIBSOVO","IDHIFA"}

#import pandas as pd

def classify_patient(row):
    # Split regimen into product list
    products = {p.strip().upper() for p in row["LOT_REGIMEN_GROUP"].split(",")}
    
    # Age calculation if available
    birth_year = row["SOURCE_PATIENT_HIPAA_BIRTH_YEAR"]
    first_dx = row["FIRST_DX"]
    age = None
    if pd.notnull(birth_year) and pd.notnull(first_dx):
        # Convert birth_year to int if it's a string
        try:
            birth_year = int(birth_year)
        except ValueError:
            birth_year = None
        
        # Extract year from datetime if it's a date
        if hasattr(first_dx, "year"):
            dx_year = first_dx.year
        else:
            dx_year = int(first_dx)
        
        if birth_year is not None:
            age = dx_year - birth_year
    
    # NEW DEFINITION (age + product). Legacy rule = product-only cohort in PATIENT COHORT above.
    # Rule 1: If at least one IC_ELIG product → IC_ELIG
    if products & IC_ELIG_PRODUCTS:
        return "IC_ELIG"
    
    # Rule 2: If only HMA products → IC_INELIG
    if products.issubset(FORCED_INELIG_PRODUCTS):
        return "IC_INELIG"
    
    # Rule 3: If age present → age<=65 → IC_ELIG else IC_INELIG
    if age is not None:
        return "IC_ELIG" if age <= 65 else "IC_INELIG"
    
    # Rule 4: Default → IC_INELIG
    return "IC_INELIG"

# Apply classification
updated_patient_intensity["PATIENT_INTENSITY_UPDATED"] = updated_patient_intensity.apply(classify_patient, axis=1)

print(updated_patient_intensity)


In [ ]:
# Define backbone hierarchies
ic_elig_hierarchy = [
    "S-DAC","HI-DAC","VANFLYTA","VYXEOS","CYCLOPHOSPHAMIDE","VINCRISTINE","FLUDARABINE",
    "IDARUBICIN","DAUNORUBICIN","MITOXANTRONE","RYDAPT","VENCLEXTA","XOSPATA","TIBSOVO",
    "REZLIDHIA","IDHIFA","MYLOTARG","DAURISMO","ONUREG","INQOVI","NEXAVAR","ETOPOSIDE",
    "CLADRIBINE","CLOFARABINE","AZACITIDINE","DECITABINE","L-DAC","ARSENIC_TRIOXIDE"
]

ic_inelig_hierarchy = [
    "VENCLEXTA","TIBSOVO","REZLIDHIA","IDHIFA","XOSPATA","DAURISMO","L-DAC",
    "AZACITIDINE","DECITABINE","INQOVI","ONUREG"
]

# Convert hierarchies to dicts for fast lookup
ic_elig_rank = {drug: rank for rank, drug in enumerate(ic_elig_hierarchy, start=1)}
ic_inelig_rank = {drug: rank for rank, drug in enumerate(ic_inelig_hierarchy, start=1)}

def get_backbone(row):
    products = [p.strip() for p in row["LOT_REGIMEN_GROUP"].split(",")]
    if row["PATIENT_INTENSITY_UPDATED"] == "IC_ELIG":
        ranks = {p: ic_elig_rank[p] for p in products if p in ic_elig_rank}
    else:  # IC_INELIG
        ranks = {p: ic_inelig_rank[p] for p in products if p in ic_inelig_rank}
    if not ranks:  # no product found in hierarchy
        return "UNKNOWN"
    return min(ranks, key=ranks.get)

# Apply function
updated_patient_intensity["BACKBONE"] = updated_patient_intensity.apply(get_backbone, axis=1)

print(updated_patient_intensity)

In [ ]:
# Renamed so the global keeps the table name used in the next cell
patient_intensity_df = spark.createDataFrame(updated_patient_intensity)
patient_intensity_df.createOrReplaceTempView("patient_intensity_based_on_age_n_product")

In [ ]:
spark.sql(''' DROP TABLE IF EXISTS {tbl}'''.format(tbl=final_patient_intensity_tbl))

spark.sql('''
CREATE TABLE {tbl} AS
(       
        SELECT * FROM patient_intensity_based_on_age_n_product
)         
          '''.format(tbl=final_patient_intensity_tbl))


#### NEW DEFINITION - PASS 2 (runs only when `intensity_definition: new`)

The rule B classes above (PATIENT_INTENSITY_UPDATED) become the patient cohort. The IC_ELIG and
IC_INELIG lines, the combined LoT and the LoT grouping are rebuilt with those classes. The SQL is the
same as the legacy cells; only the cohort changes. In a legacy run this section does nothing.

In [ ]:
if INTENSITY_DEFINITION == 'new':
    # 1. New cohort: one row per patient, same columns downstream reads (PATIENT_GID_RGMN, PATIENT_INTENSITY_GROUP).
    #    Patients without a line-1 group in the new-definition table keep their legacy class.
    spark.sql('DROP TABLE IF EXISTS {t}'.format(t=final_patient_group))
    spark.sql('''
    CREATE TABLE {new_group} AS
    SELECT L.PATIENT_GID_RGMN, L.LOW_INTENSITY, L.PATIENT_INTENSITY,
           COALESCE(N.PATIENT_INTENSITY_UPDATED, L.PATIENT_INTENSITY_GROUP) AS PATIENT_INTENSITY_GROUP,
           L.PATIENT_INTENSITY_GROUP AS PATIENT_INTENSITY_GROUP_LEGACY
    FROM {legacy_group} L
    LEFT JOIN (SELECT DISTINCT PATIENT_GID, PATIENT_INTENSITY_UPDATED FROM {new_cls}) N
      ON N.PATIENT_GID = L.PATIENT_GID_RGMN
    '''.format(new_group=final_patient_group, legacy_group=patient_group,
               new_cls=final_patient_intensity_tbl))

    # 2. Rebuild lines with the new cohort (same SQL as the legacy cells above).
    pg = final_patient_group
    ic_inelig_cons_final = spark.sql('''
SELECT *
FROM
(
    -- Smaller regimens were dropped earlier, now all the regimens are kept as it is
                                 
    SELECT *  
    --,CASE
      --  WHEN REGIMEN = 'RYDAPT' AND REGIMEN_LENGTH >= 14 THEN 1 --RYDAPT CLAIMS HAVE DOS OF 14/21 AS WELL
      --  WHEN REGIMEN_LENGTH >= 28 THEN 1
     --   ELSE 0
    --END AS SEL_FLAG
                                 
    FROM
    (   SELECT PATIENT_GID_RGMN AS PATIENT_GID,  REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE,
        --SELECT PATIENT_GID_RGMN AS PATIENT_GID, NEW_RNK, REGIMEN, LVL, REGIMEN_START_DATE, REGIMEN_END_DATE,
        DATEDIFF(REGIMEN_END_DATE, REGIMEN_START_DATE) AS REGIMEN_LENGTH
        --FROM ic_inelig_cons5
         FROM {regimen_tbl_name}
 									WHERE PATIENT_GID_RGMN IN
 									(
 										SELECT DISTINCT PATIENT_GID_RGMN
 										FROM {patient_group}
 										WHERE PATIENT_INTENSITY_GROUP = 'IC_INELIG'
 									)
    )A
)A
                                 
--WHERE SEL_FLAG = 1
                                 
'''.format(regimen_tbl_name=regimen_tbl_name, patient_group=pg))
    ic_inelig_cons_final.createOrReplaceTempView('ic_inelig_cons_final')

    spark.sql('DROP TABLE IF EXISTS {t}'.format(t=final_ic_inelig_final))
    spark.sql('''
CREATE TABLE {ic_inelig_final} AS
(	
	SELECT *,
	SUM(LINE_CHANGE) OVER (PARTITION BY PATIENT_GID ORDER BY REGIMEN_START_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS LOT
	FROM
	(
		SELECT *,
		CASE
			WHEN REG_NUM = 1 THEN 1
			WHEN PREV_GAP > 90 THEN 1
			WHEN PREV_GAP <= 90 THEN
				CASE
					WHEN BACKBONE = PREV_BACKBONE THEN 0
					WHEN BACKBONE LIKE '%VENCLEXTA%' AND PREV_BACKBONE LIKE '%HMA%' THEN 0
					WHEN BACKBONE LIKE '%HMA%' AND PREV_BACKBONE LIKE '%VENCLEXTA%' THEN 0
					ELSE 1 --WHEN BACKBONE <> PREV_BACKBONE THEN 1
				END
		END AS LINE_CHANGE
		FROM
		(
			SELECT *,
			LAG(BACKBONE) OVER (PARTITION BY PATIENT_GID ORDER BY REGIMEN_START_DATE) AS PREV_BACKBONE,
			DATEDIFF(REGIMEN_START_DATE, LAG(REGIMEN_END_DATE) OVER (PARTITION BY PATIENT_GID ORDER BY REGIMEN_START_DATE)) AS PREV_GAP
			FROM
			(
				SELECT PATIENT_GID, REGIMEN, 
				TO_DATE(REGIMEN_START_DATE) AS REGIMEN_START_DATE, 
				TO_DATE(REGIMEN_END_DATE) AS REGIMEN_END_DATE, REGIMEN_LENGTH,
				ROW_NUMBER() OVER (PARTITION BY PATIENT_GID ORDER BY REGIMEN_START_DATE) AS REG_NUM,
				CASE
          
					WHEN REGIMEN RLIKE 'VENCLEXTA' THEN 'VENCLEXTA'
					WHEN REGIMEN RLIKE 'TIBSOVO' THEN 'TIBSOVO'
					WHEN REGIMEN RLIKE 'REZLIDHIA' THEN 'REZLIDHIA'
					WHEN REGIMEN RLIKE 'IDHIFA' THEN 'IDHIFA'
					WHEN REGIMEN RLIKE 'XOSPATA' THEN 'XOSPATA'
					WHEN REGIMEN RLIKE 'DAURISMO' THEN 'DAURISMO'
					--WHEN REGIMEN RLIKE 'RYDAPT' THEN 'RYDAPT'   --REMOVING RYDAPT AND MYLOTARG FROM IC INELIG BACKBONE HIERARCHY
					--WHEN REGIMEN RLIKE 'MYLOTARG' THEN 'MYLOTARG'
					WHEN REGIMEN RLIKE 'L-DAC' THEN 'L-DAC'
					WHEN REGIMEN RLIKE 'AZACITIDINE' THEN 'HMA'
					WHEN REGIMEN RLIKE 'DECITABINE' THEN 'HMA'
					WHEN REGIMEN RLIKE 'INQOVI' THEN 'HMA'		--ADDING INQOVI AND ONUREG IN IC INELIG BACKBONE HIERARCHY
					WHEN REGIMEN RLIKE 'ONUREG' THEN 'HMA'
				END AS BACKBONE
				FROM ic_inelig_cons_final
			)A
		)A
	)
)
'''.format(ic_inelig_final=final_ic_inelig_final))

    spark.sql('DROP TABLE IF EXISTS {t}'.format(t=final_ic_elig_final))
    spark.sql('''
CREATE TABLE {ic_elig_final} AS
( 
	SELECT *,
	SUM(LINE_CHANGE) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS LOT
	FROM
	(
		SELECT *,
		CASE
			WHEN REG_NUM = 1 THEN 1
			WHEN PREV_GAP > 90 THEN 1
			WHEN PREV_GAP <= 90 THEN
				CASE
					WHEN BACKBONE = PREV_BACKBONE THEN 0	
					WHEN BACKBONE LIKE '%VENCLEXTA%' AND PREV_BACKBONE LIKE '%HMA%' THEN 0
					WHEN BACKBONE LIKE '%HMA%' AND PREV_BACKBONE LIKE '%VENCLEXTA%' THEN 0
					ELSE 1 											
				END
		END AS LINE_CHANGE
		FROM
		(
			SELECT *,
			LAG(BACKBONE) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS PREV_BACKBONE,
			DATEDIFF((REGIMEN_START_DATE), LAG(REGIMEN_END_DATE) OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE)) AS PREV_GAP
			FROM
			(
				SELECT PATIENT_GID_RGMN, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, 
				DATEDIFF((REGIMEN_END_DATE) , (REGIMEN_START_DATE)) AS REG_LEN,
				ROW_NUMBER() OVER (PARTITION BY PATIENT_GID_RGMN ORDER BY REGIMEN_START_DATE) AS REG_NUM,
				CASE
					WHEN REGIMEN LIKE '%HI-DAC%' OR REGIMEN LIKE '%S-DAC%' THEN 'S-DAC/HI-DAC'
					WHEN REGIMEN LIKE '%VANFLYTA%' THEN 'VANFLYTA'
					WHEN REGIMEN LIKE '%VYXEOS%' THEN 'VYXEOS'
					WHEN REGIMEN LIKE '%CYCLOPHOSPHAMIDE%' THEN 'CYCLOPHOSPHAMIDE'
					WHEN REGIMEN LIKE '%VINCRISTINE%' THEN 'VINCRISTINE'
					WHEN REGIMEN LIKE '%FLUDARABINE%' THEN 'FLUDARABINE'
					WHEN REGIMEN LIKE '%IDARUBICIN%' THEN 'IDARUBICIN'
					WHEN REGIMEN LIKE '%DAUNORUBICIN%' THEN 'DAUNORUBICIN'
					WHEN REGIMEN LIKE '%MITOXANTRONE%' THEN 'MITOXANTRONE'
					WHEN REGIMEN LIKE '%RYDAPT%' THEN 'RYDAPT'
					WHEN REGIMEN LIKE '%VENCLEXTA%' THEN 'VENCLEXTA'
					WHEN REGIMEN LIKE '%XOSPATA%' THEN 'XOSPATA'
					WHEN REGIMEN LIKE '%TIBSOVO%' THEN 'TIBSOVO'
					WHEN REGIMEN LIKE '%REZLIDHIA%' THEN 'REZLIDHIA'
					WHEN REGIMEN LIKE '%IDHIFA%' THEN 'IDHIFA'
					WHEN REGIMEN LIKE '%MYLOTARG%' THEN 'MYLOTARG'
					WHEN REGIMEN LIKE '%DAURISMO%' THEN 'DAURISMO'
					WHEN REGIMEN LIKE '%ONUREG%' THEN 'HMA'    --ONUREG AND INQOVI ARE TREATED AS HMAs
					WHEN REGIMEN LIKE '%INQOVI%' THEN 'HMA'
					WHEN REGIMEN LIKE '%NEXAVAR%' THEN 'NEXAVAR'
					WHEN REGIMEN LIKE '%ETOPOSIDE%' THEN 'ETOPOSIDE'
					WHEN REGIMEN LIKE '%CLADRIBINE%' THEN 'CLADRIBINE'
					WHEN REGIMEN LIKE '%CLOFARABINE%' THEN 'CLOFARABINE'
					WHEN REGIMEN LIKE '%AZACITIDINE%' OR REGIMEN LIKE '%DECITABINE%' THEN 'HMA'
					WHEN REGIMEN LIKE '%L-DAC%' THEN  'L-DAC'
					WHEN REGIMEN LIKE '%ARSENIC_TRIOXIDE%' THEN 'ARSENIC_TRIOXIDE' 
					ELSE NULL
				END AS BACKBONE
				FROM {regimen_tbl_name}
									WHERE PATIENT_GID_RGMN IN
									(
										SELECT DISTINCT PATIENT_GID_RGMN
										FROM {patient_group}
										WHERE PATIENT_INTENSITY_GROUP = 'IC_ELIG'
									)
			)A
		)A
	)	
)
'''.format(ic_elig_final=final_ic_elig_final, regimen_tbl_name=regimen_tbl_name,
                                          patient_group=pg))

    spark.sql('DROP TABLE IF EXISTS {t}'.format(t=final_pat_comb_lot))
    spark.sql('''
CREATE TABLE {pat_comb_lot} AS
(
	SELECT * 
    FROM
	(
		SELECT PATIENT_GID_RGMN AS PATIENT_GID, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, BACKBONE, PREV_BACKBONE, REG_NUM, LINE_CHANGE, LOT, 'IC_ELIG' AS TYPE
		FROM {ic_elig_final} -- SOME PATIENTS MAY HAVE ARSENIC REGIMENS

		UNION
		
		SELECT PATIENT_GID, REGIMEN, REGIMEN_START_DATE, REGIMEN_END_DATE, BACKBONE, PREV_BACKBONE, REG_NUM, LINE_CHANGE, LOT, 'IC_INELIG' AS TYPE
		FROM {ic_inelig_final}
	)   
)
'''.format(ic_elig_final=final_ic_elig_final, ic_inelig_final=final_ic_inelig_final,
                                       pat_comb_lot=final_pat_comb_lot))

    # 3. LoT grouping on the new lines (Spark equivalent of the pandas grouping above: sorted, de-duplicated
    #    union of every product in the line).
    spark.sql('DROP TABLE IF EXISTS {t}'.format(t=final_lot_regimen_group))
    spark.sql('''
    CREATE TABLE {grp} AS
    SELECT PATIENT_GID, LOT,
           MIN(REGIMEN_START_DATE) AS LOT_START_DATE,
           MAX(REGIMEN_END_DATE)   AS LOT_END_DATE,
           CONCAT_WS(', ', ARRAY_SORT(ARRAY_DISTINCT(FLATTEN(COLLECT_LIST(
               TRANSFORM(SPLIT(REGIMEN, ','), x -> TRIM(x))))))) AS LOT_REGIMEN_GROUP
    FROM {comb}
    GROUP BY PATIENT_GID, LOT
    '''.format(grp=final_lot_regimen_group, comb=final_pat_comb_lot))

    # 4. Checks: same patients as the legacy build; classes add up.
    spark.sql('''
    SELECT PATIENT_INTENSITY_GROUP_LEGACY AS LEGACY, PATIENT_INTENSITY_GROUP AS NEW_DEFINITION,
           COUNT(DISTINCT PATIENT_GID_RGMN) AS NUM_PAT
    FROM {g} GROUP BY 1, 2 ORDER BY 1, 2
    '''.format(g=final_patient_group)).show()
    spark.sql('''
    SELECT (SELECT COUNT(DISTINCT PATIENT_GID) FROM {a}) AS LEGACY_LOT_PATIENTS,
           (SELECT COUNT(DISTINCT PATIENT_GID) FROM {b}) AS NEW_LOT_PATIENTS
    '''.format(a=pat_comb_lot, b=final_pat_comb_lot)).show()
else:
    print('Legacy rule: pass 2 skipped; outputs are', pat_comb_lot, 'and', lot_regimen_group)